In [1]:
import findspark
findspark.init()

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql.functions import *

In [3]:
import os
os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages org.apache.spark:spark-sql-kafka-0-10_2.12:3.2.1 pyspark-shell'

In [4]:

spark = SparkSession.builder.master("local[3]").config("spark.jars.packages","org.apache.spark:spark-sql-kafka-0-10_2.12:3.2.1").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

In [5]:
kafkaSourceDF = spark.readStream.format("kafka").option("kafka.bootstrap.servers", "localhost:9092")\
      .option("subscribe", "trade").option("startingOffsets", "earliest").load()

In [6]:
kafkaSourceDF.printSchema()

root
 |-- key: binary (nullable = true)
 |-- value: binary (nullable = true)
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- timestampType: integer (nullable = true)



In [7]:
stockSchema = StructType([StructField("CreatedTime", StringType(),True),StructField("Type", StringType(),True),\
StructField("Amount", IntegerType(),True),StructField("BrokerCode", StringType(),True)])

In [8]:
valueDF = kafkaSourceDF.select(from_json(col("value").cast("string"), stockSchema).alias("value"))

In [9]:
valueDF.printSchema()

root
 |-- value: struct (nullable = true)
 |    |-- CreatedTime: string (nullable = true)
 |    |-- Type: string (nullable = true)
 |    |-- Amount: integer (nullable = true)
 |    |-- BrokerCode: string (nullable = true)



In [10]:
result = valueDF.select("value.*")

In [11]:
result.printSchema()

root
 |-- CreatedTime: string (nullable = true)
 |-- Type: string (nullable = true)
 |-- Amount: integer (nullable = true)
 |-- BrokerCode: string (nullable = true)



In [12]:
tradeDF = valueDF.select("value.*")\
      .withColumn("CreatedTime", to_timestamp(col("CreatedTime"), "yyyy-MM-dd HH:mm:ss"))\
      .withColumn("Buy", expr("case when Type == 'BUY' then Amount else 0 end"))\
      .withColumn("Sell", expr("case when Type == 'SELL' then Amount else 0 end"))

In [11]:
kafkaSourceDF.printSchema()

root
 |-- key: binary (nullable = true)
 |-- value: binary (nullable = true)
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- timestampType: integer (nullable = true)



In [12]:
valueDF.printSchema()

root
 |-- value: struct (nullable = true)
 |    |-- CreatedTime: string (nullable = true)
 |    |-- Type: string (nullable = true)
 |    |-- Amount: integer (nullable = true)
 |    |-- BrokerCode: string (nullable = true)



In [13]:
tradeDF.printSchema()

root
 |-- CreatedTime: timestamp (nullable = true)
 |-- Type: string (nullable = true)
 |-- Amount: integer (nullable = true)
 |-- BrokerCode: string (nullable = true)
 |-- Buy: integer (nullable = true)
 |-- Sell: integer (nullable = true)



In [14]:
windowAggDF = tradeDF.withWatermark("CreatedTime", '2 minute')\
      .groupBy(window(col("CreatedTime"), '10 minute'))\
      .agg(sum("Buy").alias("TotalBuy"),\
        sum("Sell").alias("TotalSell"))

In [16]:
outputDF = windowAggDF.select("window.start", "window.end", "TotalBuy", "TotalSell")

In [ ]:
from pyspark.sql.streaming import *
windowQuery = outputDF.writeStream\
      .format("console").outputMode("update")\
      .option("checkpointLocation", "ch12")\
      .start().awaitTermination()

In [15]:
windowAggDF.printSchema()

root
 |-- window: struct (nullable = false)
 |    |-- start: timestamp (nullable = true)
 |    |-- end: timestamp (nullable = true)
 |-- TotalBuy: long (nullable = true)
 |-- TotalSell: long (nullable = true)



In [5]:
kafkaSourceDF = spark.readStream.format("kafka").option("kafka.bootstrap.servers", "localhost:9092")\
      .option("subscribe", "first").option("startingOffsets", "earliest").load()

In [ ]:
kafkaSourceDF.writeStream.format("console").start().awaitTermination()

In [7]:
kafkaSourceDF.printSchema()

root
 |-- key: binary (nullable = true)
 |-- value: binary (nullable = true)
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- timestampType: integer (nullable = true)

